# SCIseg batch on Duke T2 — correct order, GPU-enabled, Drive-backed

**Before running:**
1. Runtime → Change runtime type → **T4 GPU** → Save.
2. Make sure **`duke_batch.zip`** is in your Google Drive (My Drive root).

**Then just run every cell top to bottom (Runtime → Run all).** Approve the Drive popup when it appears.

Watch the **`✓ Ns`** timer the loop prints per case: **~30–60 s = GPU working** 🎉; **~3–5 min = still on CPU** (tell Claude). Masks save to **My Drive/duke_masks/** and it's **resumable** (re-run after any disconnect).

### 1 — confirm GPU is attached

In [ ]:
!nvidia-smi -L   # must list a GPU; if blank, set Runtime -> T4 GPU and re-run

### 2 — install Spinal Cord Toolbox (~5–10 min; skips if already installed)

In [ ]:
import os
if not os.path.exists('/content/spinalcordtoolbox/bin/sct_deepseg'):
    !git clone --depth 1 https://github.com/spinalcordtoolbox/spinalcordtoolbox.git
    !spinalcordtoolbox/install_sct -y
os.environ['PATH'] += ':/content/spinalcordtoolbox/bin'
!sct_version

### 3 — make SCT use the GPU (swap its CPU torch for the CUDA build)

In [ ]:
import glob, subprocess
sctpy = glob.glob('/content/spinalcordtoolbox/python/envs/*/bin/python')[0]
def cuda_ok():
    r = subprocess.run([sctpy,'-c','import torch;print(torch.cuda.is_available())'],
                       capture_output=True, text=True)
    return 'True' in r.stdout
if not cuda_ok():
    ver = subprocess.run([sctpy,'-c','import torch;print(torch.__version__.split("+")[0])'],
                         capture_output=True, text=True).stdout.strip()
    print(f'Swapping SCT torch=={ver} for its CUDA (cu121) build (with deps)...')
    subprocess.run([sctpy,'-m','pip','install','--force-reinstall',
                    f'torch=={ver}','--index-url','https://download.pytorch.org/whl/cu121'])
print('SCT torch sees GPU:', cuda_ok())

### 4 — mount Google Drive (approve the popup)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### 5 — unzip the data from Drive

In [ ]:
import os, glob
!unzip -q -o /content/drive/MyDrive/duke_batch.zip -d /content/cases
os.makedirs('/content/drive/MyDrive/duke_masks', exist_ok=True)
n = len([c for c in glob.glob('/content/cases/**/*.nii.gz', recursive=True) if '_seg' not in c])
print(n, 'input scans ready')

### 6 — run SCIseg on all cases (resumable; watch the ✓ Ns timer)

In [ ]:
import os, glob, subprocess, shutil, time
os.environ['PATH'] += ':/content/spinalcordtoolbox/bin'
OUT = '/content/drive/MyDrive/duke_masks'; os.makedirs(OUT, exist_ok=True)
cases = sorted(c for c in glob.glob('/content/cases/**/*.nii.gz', recursive=True) if '_seg' not in c)
print(len(cases), 'cases\n')

def seg_one(f):
    last = None
    for cmd in (['sct_deepseg','lesion_sci_t2','-i',f],                       # SCT v7.0+
                ['sct_deepseg','-i',f,'-task','seg_sc_lesion_t2w_sci']):      # SCT v6.2-6.5
        last = subprocess.run(cmd, capture_output=True, text=True)
        if last.returncode == 0:
            return True, ''
    return False, (last.stderr or last.stdout)[-400:]

for i, f in enumerate(cases, 1):
    base = os.path.basename(f)[:-7]
    if os.path.exists(f'{OUT}/{base}_lesion_seg.nii.gz'):
        print(f'[{i}/{len(cases)}] skip (done): {base}'); continue
    t = time.time(); print(f'[{i}/{len(cases)}] {base}', flush=True)
    ok, err = seg_one(f)
    if ok:
        for m in glob.glob(f'/content/cases/{base}*_seg.nii.gz'):
            shutil.copy(m, OUT)
        print(f'    ✓ {time.time()-t:.0f}s')
    else:
        print('    ERROR:', err)
print('\nDONE. lesion masks on Drive:', len(glob.glob(f'{OUT}/*_lesion_seg.nii.gz')))

### Done
Masks are in **My Drive/duke_masks/**. Download that folder into `~/dev/group5-proto/out/` and tell Claude.

**If it disconnects:** re-run cells **2 → 6** (install skips if present, GPU-fix skips if already CUDA, loop skips done cases).